# First L0 processor example, version==0.9.0

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-607

See the associated:

  * Python module: [first_l0_processor.py](./first_l0_processor.py)
  * YAML file: [first_l0_processor.yaml](./first_l0_processor.yaml)

## Initialization

In [1]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Prefect server URL used internally: http://prefect-server:4200/api
Prefect dashboard public URL: http://localhost:4200/dashboard


In [1]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
from resources.prefect_utils import *

init_demo()
init_dask_cluster_eopf(scale=3)
# , image="ghcr.io/rs-python/rs-infrastructure-dask-eopf:feat-rspy607-l0-processing" # temp

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  
from resources.prefect_utils import * 

DependencyConflict: requested: "starlette ~= 0.13.0" but found: "starlette 0.45.3"


Auxip service: http://rs-server-adgs:8000
CADIP service: http://rs-server-cadip:8000
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
Connecting to dask gateway for 'dask-eopf': http://dask-eopf:8000 ...
Get existing dask cluster: '29b5bae95e044c6a861f09bf0f62035f'
Dask dashboard for 'dask-eopf': http://localhost:8702/clusters/29b5bae95e044c6a861f09bf0f62035f/status
Dask workers for 'dask-eopf' are up: 3/3


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.2  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


In [7]:
# Other imports
import getpass
import os
from importlib import reload
from resources import prefect_utils

# s3 bucket dirs that will contain the data
s3_base = os.path.join(
    "s3://",
    PREFECT_BLOCK_S3.bucket_name,
    PREFECT_BLOCK_S3.bucket_folder,
    "users",
    os.environ.get("RSPY_HOST_USER", getpass.getuser()),
    "l0",
)
s3_config = os.path.join(s3_base, "config")
s3_output = os.path.join(s3_base, "output")

# Upload the local configuration dir to s3 bucket
await s3_upload_dir("./l0_config", s3_config)

# For each data: 
# input_config_dir: s3 bucket folder that contains the configuration files (NOT THE VOLUMINOUS DATA !).
# It will be downloaded locally.
# payload_file: input yaml configuration file to pass to the triggering. Local to the 'input_config_dir'.
# output_data_dir: s3 bucket directory that will contain the generated data.
s1_short = {
    "input_config_dir": s3_config,
    "payload_file": "s1/iw_joborder.short.yaml",
    "output_data_dir": f"{s3_output}/s1.short",
}
s1 = {
    "input_config_dir": s3_config,
    "payload_file": "s1/iw_joborder.yaml",
    "output_data_dir": f"{s3_output}/s1",
}
s3 = {
    "input_config_dir": s3_config,
    "payload_file": "s3/s3_dordop_payload.yaml",
    "output_data_dir": f"{s3_output}/s3",
}

# Convert to json to trigger prefect flow
def to_json(my_data):
    return json.dumps(my_data).replace('"', r'\"')

16:15:51.996 | INFO    | prefect.S3Bucket - Uploading from 'l0_config/logging_config.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/logging_config.yaml'.

16:15:51.997 | INFO    | prefect.S3Bucket - Uploading from 'l0_config/s1/iw_joborder.short.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s1/iw_joborder.short.yaml'.

16:15:51.999 | INFO    | prefect.S3Bucket - Uploading from 'l0_config/s1/iw_configuration.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s1/iw_configuration.yaml'.

16:15:52.000 | INFO    | prefect.S3Bucket - Uploading from 'l0_config/s1/iw_joborder.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s1/iw_joborder.yaml'.

16:15:52.001 | INFO    | prefect.S3Bucket - Uploading from 'l0_config/s3/l0_processor_configuration.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s3/l0_processor_configuration.yaml'.

16:15:52.003 | INFO    | prefect.S3Bucket - Uploading from 'l0_config/s3/s3_dordop_payload.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s3/s3_dordop_payload.yaml'.

16:15:52.034 | INFO    | prefect.S3Bucket - Uploaded 6 files from 'l0_config' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s3/s3_dordop_payload.yaml'

s3://prefect-share/sub/dir/users/jgaucher/l0/config


In [4]:
# We use only the EOPF dask cluster in this tutorial
dask_gateway = dask_gateway_eopf
dask_client = dask_client_eopf
dask_cluster = dask_cluster_eopf
if local_mode:
    os.environ["DASK_GATEWAY_ADDRESS"] = os.environ["DASK_GATEWAY_EOPF_ADDRESS"]

# Save cluster info to be read by our flow
os.environ["DASK_CLUSTER_NAME"] = dask_cluster.name

## Deploy Prefect flow

We deploy our source code via the S3 bucket.

In [5]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{os.environ.get('RSPY_HOST_USER', getpass.getuser())}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{PREFECT_BLOCK_S3.bucket_name}/{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}'")

# Upload local directory contents
await PREFECT_BLOCK_S3.put_directory(local_path = ".", to_path = s3_code_folder)

# It doesn't follow symlinks so upload them manually
await PREFECT_BLOCK_S3.put_directory(local_path = "./resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}"

S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234
Upload local source code to: 's3://prefect-share/sub/dir/users/jgaucher/code'


In [6]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./first_l0_processor.yaml"

╭──────────────────────────────────────────────────────────────────────────────╮
│ Deployment 'first-l0-processor/sprint21-first-l0-processor' successfully     │
│ created with id 'dc0c95f1-2e87-4837-9dc7-e778cc537ce5'.                      │
╰──────────────────────────────────────────────────────────────────────────────╯

View Deployment in UI: http://prefect-server:4200/deployments/deployment/dc0c95f1-2e87-4837-9dc7-e778cc537ce5


To schedule a run for this deployment, use the following command:

        $ prefect deployment run 
'first-l0-processor/sprint21-first-l0-processor'



In [7]:
deploy_name = "first-l0-processor/sprint21-first-l0-processor"
await prefect_utils.wait_for_deployment(deploy_name)

Finished deploying prefect flow: 'first-l0-processor/sprint21-first-l0-processor'


## Run Prefect flow for S1 short data

In [12]:
output = s1_short["output_data_dir"]
print(f"Remove existing zarr products from: {output!r}")
s3_delete(output)

# Convert to json to trigger prefect flow
s1_short_str = to_json(s1_short)

Remove existing zarr products from: 's3://prefect-share/sub/dir/users/jgaucher/l0/output/s1.short'


'{\\"input_config_folder\\": \\"s3://rs-cluster-temp/stations/dpr-test/l0/input/s1\\", \\"payload_file\\": \\"iw_joborder.short.yaml\\", \\"output_data_folder\\": \\"s3://prefect-share/sub/dir/users/jgaucher/l0/output/s1.short\\"}'

In [13]:
%%bash -s "$deploy_name" "$s1_short_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

Creating flow run for deployment 
'first-l0-processor/sprint21-first-l0-processor'...
Created flow run 'debonair-gecko'.
└── UUID: a110adcc-0deb-49b0-b276-c016a659ec0e
└── Parameters: {'input_config_folder': 's3://rs-cluster-temp/stations/dpr-test/l0/input/s1', 'payload_file': 'iw_joborder.short.yaml', 'output_data_folder': 's3://prefect-share/sub/dir/users/jgaucher/l0/output/s1.short'}
└── Job Variables: {}
└── Scheduled start time: 2025-02-24 15:03:10 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/a110adcc-0deb-49b0-b276-c016a659ec0e
Watching flow run 'debonair-gecko'...


15:03:10.978 | INFO    | prefect - Flow run is in state 'Scheduled'
15:03:15.991 | INFO    | prefect - Flow run is in state 'Failed'


Flow run finished in state 'Failed'.


CalledProcessError: Command 'b'# Trigger a run for this flow from the command line\nprefect deployment run "$1" --params "$2" --watch\n'' returned non-zero exit status 1.

## Check results

In [ ]:
# Download zarr products into local
local_dir = "/tmp/zarr/"
!rm -rf "$local_dir" && mkdir -p "$local_dir"
await PREFECT_BLOCK_S3.get_directory(f"{s3_prefix_subdir}", local_dir)
!ls -al "$local_dir"

In [ ]:
# Open them with the zarr python package
# see: https://help.marine.copernicus.eu/en/articles/8077952-how-to-open-and-visualize-zarr-format-data
!pip install zarr
import zarr

for filename in s3_filenames:
    store = zarr.open(f"{local_dir}/{filename}.zarr")
    display(store.tree())
    
    # Read some data
    zarr_array = store["measurements"]["image"]["sensor1"]
    display(zarr_array)
    
    # Read the data into memory as a NumPy array
    numpy_array = zarr_array[:]
    display(numpy_array)

In [ ]:


print(f"Output zarr products will be written to: {s3_full_path}")

## 3. Shutdown the dask clusters

In [ ]:
if local_mode:

    # You can scale the clusters to 0 workers
    dask_gateway.scale_cluster(dask_cluster.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway, dask_cluster.name)

# Close the python objects
close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.